# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [ ]:
# Load the libraries as required.
import os
import pandas as pd
import requests

In [ ]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [7]:
import os
import pandas as pd
import requests

# Define file paths
data_dir = "../data/fires/"
file_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/forest-fires/forestfires.csv"
file_path = os.path.join(data_dir, "forestfires.csv")

# Create the directory if it doesn't exist
os.makedirs(data_dir, exist_ok=True)

# Download the dataset
if not os.path.exists(file_path):
    response = requests.get(file_url)
    with open(file_path, "wb") as f:
        f.write(response.content)
    print("Dataset downloaded successfully!")
else:
    print("Dataset already exists.")

# Load the dataset
df = pd.read_csv(file_path)

# Display the first few rows
df.head()


Dataset downloaded successfully!


,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0


In [8]:
import os

# Check if the file exists
print("Files in directory:", os.listdir(data_dir))

Files in directory: ['forestfires.csv']


In [9]:
# Check dataset info
df.info()

# Check for missing values
print(df.isnull().sum())

# Summary statistics
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       517 non-null    int64  
 1   Y       517 non-null    int64  
 2   month   517 non-null    object 
 3   day     517 non-null    object 
 4   FFMC    517 non-null    float64
 5   DMC     517 non-null    float64
 6   DC      517 non-null    float64
 7   ISI     517 non-null    float64
 8   temp    517 non-null    float64
 9   RH      517 non-null    int64  
 10  wind    517 non-null    float64
 11  rain    517 non-null    float64
 12  area    517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB
X        0
Y        0
month    0
day      0
FFMC     0
DMC      0
DC       0
ISI      0
temp     0
RH       0
wind     0
rain     0
area     0
dtype: int64
                X           Y        FFMC         DMC          DC         ISI  \
count  517.000000  517.000000  517.00000

In [10]:
import pandas as pd

# Load the dataset
file_path = "../data/fires/forestfires.csv"
df = pd.read_csv(file_path)

# Display the first few rows
df.head()

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0


In [11]:
# Define the feature matrix (X) and target variable (Y)
X = df.drop(columns=["area"])  # Drop target column from features
Y = df["area"]  # Target variable

# Display the first few rows of features
X.head()

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0


In [12]:
# One-hot encode categorical variables
X = pd.get_dummies(X, columns=["month", "day"], drop_first=True)

# Display transformed features
X.head()

,X,Y,FFMC,DMC,DC,ISI,temp,RH,wind,rain,...,month_may,month_nov,month_oct,month_sep,day_mon,day_sat,day_sun,day_thu,day_tue,day_wed
0,7,5,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,...,False,False,False,False,False,False,False,False,False,False
1,7,4,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,...,False,False,True,False,False,False,False,False,True,False
2,7,4,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,...,False,False,True,False,False,True,False,False,False,False
3,8,6,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,...,False,False,False,False,False,False,False,False,False,False
4,8,6,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,...,False,False,False,False,False,False,True,False,False,False


In [13]:
from sklearn.preprocessing import StandardScaler

# Identify numeric columns
numeric_cols = ["X", "Y", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"]

# Apply Standard Scaling
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# Display scaled features
X.head()

,X,Y,FFMC,DMC,DC,ISI,temp,RH,wind,rain,...,month_may,month_nov,month_oct,month_sep,day_mon,day_sat,day_sun,day_thu,day_tue,day_wed
0,1.008313,0.569860,-0.805959,-1.323326,-1.830477,-0.860946,-1.842640,0.411724,1.498614,-0.073268,...,False,False,False,False,False,False,False,False,False,False
1,1.008313,-0.244001,-0.008102,-1.179541,0.488891,-0.509688,-0.153278,-0.692456,-1.741756,-0.073268,...,False,False,True,False,False,False,False,False,True,False
2,1.008313,-0.244001,-0.008102,-1.049822,0.560715,-0.509688,-0.739383,-0.692456,-1.518282,-0.073268,...,False,False,True,False,False,True,False,False,False,False
3,1.440925,1.383722,0.191362,-1.212361,-1.898266,-0.004756,-1.825402,3.233519,-0.009834,0.603155,...,False,False,False,False,False,False,False,False,False,False
4,1.440925,1.383722,-0.243833,-0.931043,-1.798600,0.126966,-1.291012,3.356206,-1.238940,-0.073268,...,False,False,False,False,False,False,True,False,False,False


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [8]:
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np


In [9]:
# Identify numeric and categorical columns
numeric_features = ["X", "Y", "FFMC", "DMC", "DC", "ISI", "temp", "RH", "wind", "rain"]
categorical_features = ["month", "day"]

In [10]:
preproc1 = ColumnTransformer([
    ("num_scaler", StandardScaler(), numeric_features),
    ("cat_encoder", OneHotEncoder(drop="first"), categorical_features)
])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, MinMaxScaler, OneHotEncoder

# Define your feature lists
numeric_features = ["your_numeric_feature_1", "your_numeric_feature_2"]  # Replace with actual numeric feature names
categorical_features = ["your_categorical_feature_1", "your_categorical_feature_2"]  # Replace with actual categorical feature names

# Define the preprocessing pipeline
preproc2 = ColumnTransformer([
    ("num_transform", Pipeline([
        ("power", PowerTransformer(method="yeo-johnson")),  # Non-linear transformation
        ("scaler", MinMaxScaler())  # Scaling after transformation
    ]), numeric_features),
    ("cat_encoder", OneHotEncoder(drop="first"), categorical_features)
])


In [14]:
preproc2 = ColumnTransformer([
    ("num_transform", Pipeline([
        ("power", PowerTransformer(method="yeo-johnson")),  # Non-linear transformation
        ("scaler", MinMaxScaler())  # Scaling after transformation
    ]), numeric_features),
    ("cat_encoder", OneHotEncoder(drop="first"), categorical_features)
])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [5]:
# Pipeline A = preproc1 + baseline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [16]:
# Baseline model (e.g., Lasso Regression)
baseline_model = Pipeline([
    ("preprocessing", preproc2),  # Assign preproc2 from the previous section
    ("regressor", Lasso())  # Choose Lasso as the baseline model
])

# Advanced model (e.g., RandomForestRegressor)
advanced_model = Pipeline([
    ("preprocessing", preproc2),  # Assign preproc2 from the previous section
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))  # More advanced regressor
])



# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score
import numpy as np


In [28]:
import pandas as pd

# Load the dataset
file_path = "../data/fires/forestfires.csv"  # Adjust the path if necessary
data = pd.read_csv(file_path)

# Display first few rows
print(data.head())


   X  Y month  day  FFMC   DMC     DC  ISI  temp  RH  wind  rain  area
0  7  5   mar  fri  86.2  26.2   94.3  5.1   8.2  51   6.7   0.0   0.0
1  7  4   oct  tue  90.6  35.4  669.1  6.7  18.0  33   0.9   0.0   0.0
2  7  4   oct  sat  90.6  43.7  686.9  6.7  14.6  33   1.3   0.0   0.0
3  8  6   mar  fri  91.7  33.3   77.5  9.0   8.3  97   4.0   0.2   0.0
4  8  6   mar  sun  89.3  51.3  102.2  9.6  11.4  99   1.8   0.0   0.0


In [29]:
print(data.columns)


Index(['X', 'Y', 'month', 'day', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH',
       'wind', 'rain', 'area'],
      dtype='object')


In [31]:
# Define features (X) and target (y)
X = data.drop(columns=['area'])  # Features
y = data['area']  # Target variable


In [33]:
import pandas as pd

# Load dataset
data = pd.read_csv('../data/fires/forestfires.csv')  # Replace with actual dataset

# Define features and target variable
X = data.drop(columns=['area'])  # Replace 'target' with actual target column name
y = data['area']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [34]:
# Baseline: Lasso Regression
lasso_pipeline = Pipeline([
    ("preprocessing", preproc2),  # Use preproc2 from previous section
    ("regressor", Lasso())
])

# Ridge Regression Pipeline
ridge_pipeline = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", Ridge())
])

# Random Forest Pipeline (More Advanced)
rf_pipeline = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", RandomForestRegressor(random_state=42))
])

# Gradient Boosting Pipeline (More Advanced)
gb_pipeline = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", GradientBoostingRegressor(random_state=42))
])


In [35]:
lasso_param_grid = {
    "regressor__alpha": [0.001, 0.01, 0.1, 1]  # Tuning regularization strength
}

ridge_param_grid = {
    "regressor__alpha": [0.1, 1, 10, 100]  # Tuning regularization strength
}

rf_param_grid = {
    "regressor__n_estimators": [50, 100, 200, 300],  # Number of trees in the forest
}

gb_param_grid = {
    "regressor__learning_rate": [0.01, 0.05, 0.1, 0.2],  # Learning rate tuning
}


# Evaluate

+ Which model has the best performance?

In [48]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, MinMaxScaler

# Define numeric and categorical feature lists
numeric_features = ['X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']
categorical_features = ['month', 'day']  # The non-numeric columns causing issues

# Updated ColumnTransformer with OneHotEncoder
preproc2 = ColumnTransformer([
    ("num_transform", Pipeline([
        ("power", PowerTransformer(method="yeo-johnson")),  # Apply power transform
        ("scaler", MinMaxScaler())  # Scale numeric features
    ]), numeric_features),  # Transform only numeric features

    ("cat_encoder", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)  # One-hot encode categorical features
])


In [49]:
X_train = pd.get_dummies(X_train, columns=['month', 'day'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['month', 'day'], drop_first=True)

print("Updated X_train:\n", X_train.head())  # Check if encoding worked


Updated X_train:
      X  Y  FFMC    DMC     DC   ISI  temp  RH  wind  rain  ...  month_may  \
329  4  3  92.2  102.3  751.5   8.4  23.5  27   4.0   0.0  ...      False   
173  4  4  90.9  126.5  686.5   7.0  17.7  39   2.2   0.0  ...      False   
272  2  5  92.1  152.6  658.2  14.3  20.2  47   4.0   0.0  ...      False   
497  3  4  96.1  181.1  671.2  14.3  32.3  27   2.2   0.0  ...      False   
182  5  4  86.8   15.6   48.3   3.9  12.4  53   2.2   0.0  ...      False   

     month_nov  month_oct  month_sep  day_mon  day_sat  day_sun  day_thu  \
329      False      False       True    False     True    False    False   
173      False      False       True     True    False    False    False   
272      False      False      False    False    False    False    False   
497      False      False      False    False    False    False    False   
182      False      False      False    False    False     True    False   

     day_tue  day_wed  
329    False    False  
173    False  

In [50]:
print("Updated data types in X_train:\n", X_train.dtypes)


Updated data types in X_train:
 X              int64
Y              int64
FFMC         float64
DMC          float64
DC           float64
ISI          float64
temp         float64
RH             int64
wind         float64
rain         float64
month_aug       bool
month_dec       bool
month_feb       bool
month_jan       bool
month_jul       bool
month_jun       bool
month_mar       bool
month_may       bool
month_nov       bool
month_oct       bool
month_sep       bool
day_mon         bool
day_sat         bool
day_sun         bool
day_thu         bool
day_tue         bool
day_wed         bool
dtype: object


In [51]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)

print("Lasso Model Fitted Successfully!")


Lasso Model Fitted Successfully!


In [52]:
print("Data types in X_train:\n", X_train.dtypes)

# Find non-numeric columns
non_numeric_columns = X_train.select_dtypes(exclude=['number']).columns
print("\nNon-numeric columns:", non_numeric_columns.tolist())


Data types in X_train:
 X              int64
Y              int64
FFMC         float64
DMC          float64
DC           float64
ISI          float64
temp         float64
RH             int64
wind         float64
rain         float64
month_aug       bool
month_dec       bool
month_feb       bool
month_jan       bool
month_jul       bool
month_jun       bool
month_mar       bool
month_may       bool
month_nov       bool
month_oct       bool
month_sep       bool
day_mon         bool
day_sat         bool
day_sun         bool
day_thu         bool
day_tue         bool
day_wed         bool
dtype: object

Non-numeric columns: ['month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul', 'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct', 'month_sep', 'day_mon', 'day_sat', 'day_sun', 'day_thu', 'day_tue', 'day_wed']


In [54]:
X_train = X_train.astype(float)  # Convert all columns to float
X_test = X_test.astype(float)  # Ensure test data is the same format

print("Updated X_train types:\n", X_train.dtypes)  # Check if everything is numeric


Updated X_train types:
 X            float64
Y            float64
FFMC         float64
DMC          float64
DC           float64
ISI          float64
temp         float64
RH           float64
wind         float64
rain         float64
month_aug    float64
month_dec    float64
month_feb    float64
month_jan    float64
month_jul    float64
month_jun    float64
month_mar    float64
month_may    float64
month_nov    float64
month_oct    float64
month_sep    float64
day_mon      float64
day_sat      float64
day_sun      float64
day_thu      float64
day_tue      float64
day_wed      float64
dtype: object


In [55]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)  # Simple Lasso model
lasso.fit(X_train, y_train)  # Fit the model

print("Lasso Model Fitted Successfully!")


Lasso Model Fitted Successfully!


In [58]:
from sklearn.linear_model import Lasso

# Initialize and fit the Lasso model
lasso = Lasso(alpha=0.1)  # Adjust alpha if needed
lasso.fit(X_train, y_train)

print("✅ Lasso Model Fitted Successfully!")


✅ Lasso Model Fitted Successfully!


# Export

+ Save the best performing model to a pickle file.

In [59]:
# Store model scores in a dictionary
model_scores = {
    "Lasso": grid_lasso.best_score_ if hasattr(grid_lasso, "best_score_") else float('-inf'),
    "Ridge": grid_ridge.best_score_ if hasattr(grid_ridge, "best_score_") else float('-inf'),
    "Random Forest": grid_rf.best_score_ if hasattr(grid_rf, "best_score_") else float('-inf'),
    "Gradient Boosting": grid_gb.best_score_ if hasattr(grid_gb, "best_score_") else float('-inf')
}

# Find the model with the highest R² score
best_model_name = max(model_scores, key=model_scores.get)
best_model_score = model_scores[best_model_name]

print(f" Best Performing Model: {best_model_name} with R² Score of {best_model_score:.4f}")


 Best Performing Model: Lasso with R² Score of -inf


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.